# Feature Selection via Lasso Regression

In this notebook, we perform feature selection using an L1 regularization technique (Lasso Regression). 
Our strategy is to:
1. Use our custom, from-scratch Lasso Regression implementation.
2. Perform hyperparameter tuning to find the ideal regularization penalty (`alpha`) by minimizing Mean Squared Error (MSE) on a validation set.
3. Observe which feature weights are driven to zero by the L1 penalty, effectively identifying and dropping the unnecessary features.

In [100]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from utils import lasso_regression, mse

In [101]:
seed = 46
df = sns.load_dataset('tips')
df

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4
...,...,...,...,...,...,...,...
239,29.03,5.92,Male,No,Sat,Dinner,3
240,27.18,2.00,Female,Yes,Sat,Dinner,2
241,22.67,2.00,Male,Yes,Sat,Dinner,2
242,17.82,1.75,Male,No,Sat,Dinner,2


### Encode Categorical Data
Machine learning algorithms require numerical input. Here we encode our categorical variables into numeric formats using pandas `cat.codes` and One-Hot Encoding (`get_dummies`).

In [102]:
df['sex'] = df['sex'].cat.codes
df['smoker'] = df['smoker'].cat.codes
df['time'] = df['time'].cat.codes
df = pd.get_dummies(data=df, columns=['day'])

# Split and Standardize Data
To properly evaluate our model and tune our hyperparameters without data leakage, we split our data into three sets: Training (60%), Validation (20%), and Test (20%). 
We also apply `StandardScaler` to ensure all features are on the same scale, which is a strict requirement for regularized models like Lasso.

In [103]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


def produce_training_data(df, features):
  X = df[features]
  y = df['tip'].to_numpy()

  X_train, X_, y_train, y_ = train_test_split(X, y, test_size=0.4, random_state=seed)
  X_test, X_validate, y_test, y_validate = train_test_split(X_, y_, test_size=0.5, random_state=seed)

  scaler = StandardScaler()
  X_train_standard = scaler.fit_transform(X_train)
  X_validate_scaled = scaler.transform(X_validate)

  assert X_train_standard.shape == X_train.shape, "Standardized X_train shape does not match X_train's shape"
  assert X_validate_scaled.shape == X_validate.shape, "Standardized X_train shape does not match X_train's shape"
  assert X_train.shape[0] == y_train.shape[0], "X train's shape does not match the y train's shape"


  X_train_standard = np.concat((np.ones(X_train_standard.shape[0]).reshape(-1, 1), X_train_standard), axis=1)
  X_validate_scaled = np.concat((np.ones(X_validate_scaled.shape[0]).reshape(-1, 1), X_validate_scaled), axis=1)

  return X_train_standard, X_validate_scaled, y_train, y_validate


In [104]:
# Feature selection with Lasso regression

features = df.columns.drop(['tip'])
X_train, X_validate, y_train, y_validate = produce_training_data(df, features)

# Goal: Hyperparameter Tuning (Finding the Best Alpha)
We need an algorithm that iterates through a list of potential `alpha` values, trains a Lasso model for each, evaluates the predictions on the validation set, and returns the `alpha` that yields the lowest Mean Squared Error (MSE).

In [105]:
def find_ideal_alpha(alphas):
  result = []
  for alpha in alphas:
    weights = lasso_regression(X_train, y_train, n_iterations=10000, alpha=alpha)
    pred = X_validate @ weights
    result.append([
      alpha,
      mse(y_validate, pred)
    ])

  result = np.array(result)
  ideal_alpha = result[result[:, 1].argmin()][0]
  return ideal_alpha

### Step 1: Broad Search
First, we search for the ideal `alpha` across a broad, logarithmic range (e.g., 0.0001, 0.001, 0.01, 0.1, 1) to identify the general magnitude of the penalty.

In [106]:
ideal_alpha = find_ideal_alpha([0.0001 * 10 ** p for p in range(7)])
ideal_alpha
# Lowest MSE for alpha = 0.1
# 0.01 < ideal alpha < 1

np.float64(0.1)

### Step 2: Narrowing the Search Space
The broad search indicated that the best `alpha` is around `0.1`. 
This means the true ideal `alpha` likely sits somewhere between `0.01` and `1.0`. Next, we perform a slightly more granular linear search across this narrowed window.

In [107]:
ideal_alpha = find_ideal_alpha([*[1 - 0.1 * p for p in range(10)], *[0.1 - 0.01 * p for p in range(10)]])
ideal_alpha

np.float64(0.19999999999999996)

### Step 3: Fine-Tuning
The previous step narrowed the ideal `alpha` down to roughly `0.2`. 
To be more precise, we will search with an additional decimal point of precision around this area (between `0.1` and `0.3`).

In [108]:
ideal_alpha = find_ideal_alpha([ideal_alpha + 0.1 - 0.01 * p for p in range(20)])
ideal_alpha = round(ideal_alpha, 2)
ideal_alpha

np.float64(0.18)

### Finalizing Feature Selection
- With two decimals of precision, `0.18` is our optimal `alpha`.
- We now train our final Lasso Regression model on the training data using `alpha = 0.18` to see which features are penalized out of the model.
- **Note on Convergence**: Because our custom gradient descent solver can struggle to fully converge at the non-differentiable corner of the L1 penalty, some 'dead' features might have weights hovering infinitesimally close to zero rather than exactly `0.0`. To manually enforce the sparsity that Lasso is known for, we round the weights to 2 decimal places. Features with a weight of `0.00` are effectively dropped.

In [109]:
weights = lasso_regression(X_train, y_train, n_iterations=10000, alpha=ideal_alpha)
weights = np.round(weights, 2)
weights

# NOTE: Because the solver failed to fully converge at the non-differentiable corner of the L1 penalty, some 'dead' features have weights hovering infinitesimally close to zero. 
#       I am rounding to 2 decimals to manually enforce the clear sparsity that the algorithm failed to finalize.

array([ 2.79,  0.79, -0.  , -0.  ,  0.  ,  0.  ,  0.  ,  0.  , -0.  ,
        0.  ])

In [110]:
weight_names = ['B' , *features]

dict(zip(weight_names, weights))

{'B': np.float64(2.79),
 'total_bill': np.float64(0.79),
 'sex': np.float64(-0.0),
 'smoker': np.float64(-0.0),
 'time': np.float64(0.0),
 'size': np.float64(0.0),
 'day_Thur': np.float64(0.0),
 'day_Fri': np.float64(0.0),
 'day_Sat': np.float64(-0.0),
 'day_Sun': np.float64(0.0)}

### Conclusion
By looking at the final rounded weights, we can see which features the Lasso Regression model deemed important and which were driven to `0.00`. This provides us with a mathematical confirmation of our visual findings from the Multivariate Analysis step!